# Phase 6.0-B: 疾病关联网络分析

**项目**: Human LncRNA Atlas

**分析目标**: 构建疾病-lncRNA-靶基因三层网络，识别潜在治疗靶点

**数据源**: `/api/v1/export/disease-network` API

---

## 分析流程

1. 疾病网络数据获取
2. 三层网络构建（疾病-基因-lncRNA）
3. 网络拓扑分析
4. 重大疾病的 lncRNA 靶点识别
5. 潜在治疗靶点排行榜
6. 疾病间共享 lncRNA 分析

In [ ]:
# 环境准备
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import requests
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')
sns.set_palette("tab10")

API_BASE_URL = "http://localhost:8000/api/v1"
print("✅ 环境准备完成")

In [ ]:
# 重大疾病清单
major_diseases = [
    'diabetes',
    'alzheimer',
    'cancer',
    'cardiovascular',
    'autism'
]

# 获取所有疾病网络数据
all_networks = {}

for disease in major_diseases:
    response = requests.get(
        f"{API_BASE_URL}/export/disease-network",
        params={"trait_name": disease, "limit": 5000}
    )
    
    if response.status_code == 200:
        data = response.json()
        all_networks[disease] = data
        print(f"✅ {disease}: {len(data['nodes'])} 节点, {len(data['edges'])} 边")
    else:
        print(f"❌ {disease}: 请求失败")

print(f"\n总计获取 {len(all_networks)} 个疾病网络")

## 2. 三层网络构建示例（以糖尿病为例）

In [ ]:
# 选择糖尿病网络进行详细分析
diabetes_data = all_networks.get('diabetes', {})

if diabetes_data:
    # 构建 NetworkX 图
    G_diabetes = nx.Graph()
    
    # 添加节点（带类型属性）
    for node in diabetes_data['nodes']:
        G_diabetes.add_node(
            node['id'],
            name=node['name'],
            node_type=node['type']
        )
    
    # 添加边
    for edge in diabetes_data['edges']:
        G_diabetes.add_edge(
            edge['source'],
            edge['target'],
            edge_type=edge['type']
        )
    
    print("糖尿病调控网络统计:")
    print(f"  - 总节点数: {G_diabetes.number_of_nodes()}")
    print(f"  - 疾病节点: {sum(1 for n, d in G_diabetes.nodes(data=True) if d['node_type']=='disease')}")
    print(f"  - 基因节点: {sum(1 for n, d in G_diabetes.nodes(data=True) if d['node_type']=='gene')}")
    print(f"  - lncRNA 节点: {sum(1 for n, d in G_diabetes.nodes(data=True) if d['node_type']=='lncrna')}")
    print(f"  - 总边数: {G_diabetes.number_of_edges()}")
    print(f"  - 网络密度: {nx.density(G_diabetes):.6f}")

In [ ]:
# 网络中心性分析
degree_cent = nx.degree_centrality(G_diabetes)
betweenness_cent = nx.betweenness_centrality(G_diabetes)

# 识别关键 lncRNA 节点
lncrna_centrality = [
    {
        'node_id': node,
        'name': data['name'],
        'degree_centrality': degree_cent[node],
        'betweenness_centrality': betweenness_cent[node]
    }
    for node, data in G_diabetes.nodes(data=True)
    if data['node_type'] == 'lncrna'
]

lncrna_df = pd.DataFrame(lncrna_centrality).sort_values(
    'betweenness_centrality', ascending=False
)

print("=" * 80)
print("糖尿病关联 lncRNA 中心性排行（Top 20）")
print("=" * 80)
print(lncrna_df.head(20))

# 保存
lncrna_df.to_excel('results/diabetes_lncrna_targets.xlsx', index=False)
print("\n✅ 糖尿病 lncRNA 靶点已保存")

## 3. 潜在治疗靶点排行榜

In [ ]:
# 综合评分：度中心性 × 介数中心性
lncrna_df['composite_score'] = (
    lncrna_df['degree_centrality'] * lncrna_df['betweenness_centrality']
)

top_targets = lncrna_df.nlargest(30, 'composite_score')
top_targets['Rank'] = range(1, 31)

print("潜在治疗靶点排行榜（Top 30）:")
print(top_targets[['Rank', 'name', 'degree_centrality', 
                   'betweenness_centrality', 'composite_score']].head(15))

# 可视化
plt.figure(figsize=(12, 8))
plt.barh(top_targets['name'].head(20), top_targets['composite_score'].head(20),
        color=sns.color_palette('rocket_r', 20), edgecolor='black')
plt.xlabel('Composite Score (Degree × Betweenness)', fontsize=12)
plt.title('Top 20 Potential Therapeutic Targets for Diabetes', 
         fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('figures/14_diabetes_therapeutic_targets.png', dpi=300, bbox_inches='tight')
print("✅ 治疗靶点排行图已保存")
plt.show()

## 4. 多疾病 lncRNA 重叠分析

In [ ]:
# 提取每个疾病的 lncRNA 集合
disease_lncrnas = {}

for disease, network_data in all_networks.items():
    lncrnas = set(
        node['name'] for node in network_data['nodes']
        if node['type'] == 'lncrna'
    )
    disease_lncrnas[disease] = lncrnas
    print(f"{disease}: {len(lncrnas)} lncRNAs")

# 计算疾病间共享的 lncRNA
disease_names = list(disease_lncrnas.keys())
n_diseases = len(disease_names)
shared_lncrna_matrix = np.zeros((n_diseases, n_diseases), dtype=int)

for i, d1 in enumerate(disease_names):
    for j, d2 in enumerate(disease_names):
        if i == j:
            shared_lncrna_matrix[i, j] = len(disease_lncrnas[d1])
        else:
            shared = disease_lncrnas[d1] & disease_lncrnas[d2]
            shared_lncrna_matrix[i, j] = len(shared)

shared_disease_df = pd.DataFrame(
    shared_lncrna_matrix,
    index=disease_names,
    columns=disease_names
)

print("\n疾病间共享 lncRNA 矩阵:")
print(shared_disease_df)

In [ ]:
# 疾病间共享热力图
plt.figure(figsize=(10, 8))

sns.heatmap(
    shared_disease_df,
    annot=True,
    fmt='d',
    cmap='Blues',
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Shared lncRNA Count'},
    annot_kws={'fontsize': 11, 'fontweight': 'bold'}
)

plt.title('Shared lncRNAs Between Diseases', 
         fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Disease', fontsize=12)
plt.ylabel('Disease', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('figures/15_disease_shared_lncrnas.png', dpi=300, bbox_inches='tight')
print("✅ 疾病共享热力图已保存")
plt.show()

## 5. 网络可视化（糖尿病子网络）

In [ ]:
# 绘制糖尿病三层网络（限制规模以提高可读性）
# 选择 Top 20 lncRNA 及其连接
top_20_lncrnas = lncrna_df.head(20)['node_id'].tolist()
subgraph_nodes = set(top_20_lncrnas)

# 添加这些 lncRNA 连接的基因和疾病节点
for node in top_20_lncrnas:
    subgraph_nodes.update(G_diabetes.neighbors(node))

G_sub = G_diabetes.subgraph(subgraph_nodes).copy()

plt.figure(figsize=(16, 14))

# 布局
pos = nx.spring_layout(G_sub, k=3, iterations=50, seed=42)

# 节点颜色：按类型
node_colors = []
for node in G_sub.nodes():
    node_type = G_sub.nodes[node]['node_type']
    if node_type == 'disease':
        node_colors.append('red')
    elif node_type == 'gene':
        node_colors.append('lightblue')
    else:  # lncrna
        node_colors.append('lightgreen')

# 节点大小：根据度
node_sizes = [300 + 1000 * degree_cent.get(node, 0) for node in G_sub.nodes()]

# 绘制
nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors, node_size=node_sizes,
                      alpha=0.8, edgecolors='black', linewidths=1.5)
nx.draw_networkx_edges(G_sub, pos, alpha=0.3, edge_color='gray', width=1)
nx.draw_networkx_labels(G_sub, pos, font_size=7, font_weight='bold')

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', edgecolor='black', label='Disease'),
    Patch(facecolor='lightblue', edgecolor='black', label='Gene'),
    Patch(facecolor='lightgreen', edgecolor='black', label='lncRNA')
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=13)

plt.title('Diabetes Three-Layer Network (Disease-Gene-lncRNA)', 
         fontsize=16, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/16_diabetes_network_visualization.png', dpi=300, bbox_inches='tight')
print("✅ 网络可视化已保存")
plt.show()

## 6. 关键发现总结

In [ ]:
print("=" * 80)
print("疾病关联网络分析 - 关键发现")
print("=" * 80)

print(f"\n1. 疾病网络规模")
for disease, data in all_networks.items():
    print(f"   - {disease}: {len(data['nodes'])} 节点, {len(data['edges'])} 边")

print(f"\n2. 糖尿病网络特征")
print(f"   - 网络密度: {nx.density(G_diabetes):.6f}")
print(f"   - Top lncRNA: {lncrna_df.iloc[0]['name']}")
print(f"     (介数中心性: {lncrna_df.iloc[0]['betweenness_centrality']:.4f})")

print(f"\n3. 疾病间共享")
if len(disease_names) >= 2:
    max_shared = 0
    max_pair = ('', '')
    for i in range(len(disease_names)):
        for j in range(i+1, len(disease_names)):
            shared = shared_disease_df.iloc[i, j]
            if shared > max_shared:
                max_shared = shared
                max_pair = (disease_names[i], disease_names[j])
    print(f"   - 最大共享: {max_pair[0]}-{max_pair[1]} ({max_shared} lncRNAs)")

print("\n" + "=" * 80)
print("分析完成！")
print("=" * 80)